# Sjöbrisprognos för tävlingssegling

> Minimal clean-version: samma huvudflöde som originalet, men med tydligare kommentarer och bara små strukturella städningar.

## Översikt

Den här versionen bygger vidare direkt på originalnotebooken, men huvudflödet är nu:

1. Läs in ERA5 för en punkt vid `33.708965, -118.268343` för åren 2010-2015.
2. Läs in GFS för samma punkt med samma typ av inläsningsstruktur som i originalet.
3. Skapa punktbaserade fysikfeatures och en enkel ERA5-baserad sjöbrisflagga.
4. Filtrera bort alla historiska ERA5-dagar som inte kvalificerar som sjöbrisdagar.
5. Ta GFS-indatan kl. 10 lokal tid i Los Angeles och jämför mot de historiska ERA5-fall som är sjöbrisfall.
6. Skapa prognos från de vanligaste historiska utfallen mellan 11-16 lokal tid.

## Återanvänt från originalet

- samma struktur för `download_era5_year(...)`
- samma grundstruktur för `open_era5(...)`
- samma GFS-nedladdningsflöde
- samma robusta GFS-GRIB-matchning som grund
- samma vind- och tidshjälpare som bas

## Ersatt i den här versionen

- LSTM-dataset, träningsloop, lossfunktioner och modellinferens
- klassisk train/val/test för sekvensmodell

De delarna är ersatta av:

- punktbaserat historikbibliotek
- filtrering till endast ERA5-fall med sjöbris
- analogmatchning mot GFS kl. 10 lokal tid
- prognos från vanligaste historiska utfall

In [1]:
# Installerar paket som behövs för datanedladdning, bearbetning, visualisering och PyTorch-träning.
# Kör denna cell först i en ny miljö (t.ex. Colab). Om paketen redan finns installerade går den snabbt igenom.
!pip -q install cdsapi cfgrib eccodes xarray pandas numpy matplotlib scikit-learn torch joblib requests


## Konfiguration av CDS API
Sätt `CDSAPI_URL` och `CDSAPI_KEY` som miljövariabler i stället för att lägga dem i notebooken.

### Tips: CDSAPI_KEY som miljövariabel

Notebooken förutsätter att `CDSAPI_KEY` redan finns i miljön innan ERA5-cellen körs.

**I en notebook-session (temporärt):**
```python
import os
os.environ["CDSAPI_KEY"] = "DIN_UID:DIN_API_NYCKEL"
```

**I Linux/macOS-terminal:**
```bash
export CDSAPI_KEY="DIN_UID:DIN_API_NYCKEL"
```

**I Windows PowerShell:**
```powershell
$env:CDSAPI_KEY="DIN_UID:DIN_API_NYCKEL"
```


In [ ]:
# Läser CDS API-uppgifter från miljövariabler så att ERA5 kan laddas ned säkert utan att nyckeln skrivs direkt i notebooken.
import os

# Standard-URL för Copernicus CDS API.
CDSAPI_URL = os.getenv("CDSAPI_URL", "https://cds.climate.copernicus.eu/api")

# API-nyckeln måste finnas i miljön innan ERA5-nedladdningen körs.
CDSAPI_KEY = os.getenv("CDSAPI_KEY", "")

# Säkerställ att URL finns i miljön även om användaren inte satte den manuellt.
os.environ["CDSAPI_URL"] = CDSAPI_URL

if not CDSAPI_KEY:
    raise EnvironmentError(
        "CDSAPI_KEY saknas. Sätt den som miljövariabel innan du kör ERA5-nedladdningen."
    )

# Sätt tillbaka nyckeln i miljön så att cdsapi-klienten hittar den.
os.environ["CDSAPI_KEY"] = CDSAPI_KEY
print("CDS API-konfiguration laddad från miljövariabler.")

In [3]:
# Importer, grundinställningar och sökvägar.
# Denna cell bygger vidare på originalets konfigurationscell men är nu anpassad
# till en enda plats och ett analogbaserat arbetsflöde.
import os
import glob
import warnings
import requests
from urllib.parse import urlencode
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cdsapi

from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# =========================
# GLOBAL CONFIG – ändra här
# =========================
LOCAL_TZ = "America/Los_Angeles"

# Punkt som prognosen fokuserar på.
TARGET_POINT_LAT = 33.708965
TARGET_POINT_LON = -118.268343

# Liten hämtbox runt punkten för ERA5/GFS. Punktserien extraheras sedan med närmaste gridpunkt.
POINT_BUFFER_DEG = 0.40
DOWNLOAD_AREA = [
    TARGET_POINT_LAT + POINT_BUFFER_DEG,
    TARGET_POINT_LON - POINT_BUFFER_DEG,
    TARGET_POINT_LAT - POINT_BUFFER_DEG,
    TARGET_POINT_LON + POINT_BUFFER_DEG,
]

# ERA5-period som användaren efterfrågade.
TRAIN_YEARS = [str(y) for y in range(2010, 2011)]
TARGET_MONTH = ["05", "06", "07", "08", "09"]
TARGET_DAYS = [str(z) for z in range(1, 32)]
ERA5_DATA_VERSION = "v1"

USE_RDA_GFS = True
RDA_GFS_START_DATE = "20250601"
RDA_GFS_END_DATE = "20250618"
RDA_GFS_CYCLE = "00"
GFS_FORECAST_HOURS = [f"{h:03d}" for h in range(0, 25, 3)]



# Analogflöde.
ANALOG_MATCH_LOCAL_HOUR = 9

USE_RDA_GFS = True
RDA_GFS_START_DATE = "20250601"
RDA_GFS_END_DATE = "20250618"
RDA_GFS_CYCLE = "00"
FORCE_REDOWNLOAD_GFS = False

# Bara timmar nära 09 lokal tid i Los Angeles
GFS_FORECAST_HOURS = ["015", "018"]

FORECAST_LOCAL_HOURS = [11, 12, 13, 14, 15, 16]
ANALOG_TOP_K = 20
ANALOG_DISTANCE_METHOD = "euclidean"  # "euclidean" eller "cosine"

# Enkel sjöbrisklassning på dagsnivå, baserad på punktens ERA5-utveckling.
SEA_BREEZE_MIN_SPEED_INCREASE = 1.5
SEA_BREEZE_MIN_DIR_SHIFT_DEG = 25.0
SEA_BREEZE_ONSHORE_DIR_MIN = 180.0
SEA_BREEZE_ONSHORE_DIR_MAX = 260.0
SEA_BREEZE_MIN_TEMP_RISE_C = 1.0

# GFS-prognostimmar som laddas ned från samma körning.
GFS_MAX_LOOKBACK_HOURS = 48
GFS_MIN_FILES_FOR_RUN = 8

# Om None används första tillgängliga lokala prognosdygnet med kl 10-data.
SELECTED_GFS_DATE = None

# Kataloger för indata och resultat.
ERA5_DIR = "/content/era5_point_analog"
GFS_DIR = "/content/gfs_point_analog"
OUTPUT_DIR = "/content/output_point_analog"

os.makedirs(ERA5_DIR, exist_ok=True)
os.makedirs(GFS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Target point:", TARGET_POINT_LAT, TARGET_POINT_LON)
print("ERA5 years:", TRAIN_YEARS)
print("Download area:", DOWNLOAD_AREA)
print("Analog match hour (local):", ANALOG_MATCH_LOCAL_HOUR)
print("Forecast hours (local):", FORECAST_LOCAL_HOURS)

Target point: 33.708965 -118.268343
ERA5 years: ['2010']
Download area: [34.108965, -118.66834300000001, 33.308965, -117.868343]
Analog match hour (local): 9
Forecast hours (local): [11, 12, 13, 14, 15, 16]


In [4]:
# Hjälpfunktioner för vindberäkningar, tidskonvertering, geografi och punktuttag.

def wind_speed(u, v):
    return np.sqrt(u**2 + v**2)

def wind_dir_deg(u, v):
    return (270 - np.degrees(np.arctan2(v, u))) % 360

def circular_diff_deg(a, b):
    return (a - b + 180) % 360 - 180

def circular_mean_deg(values, weights=None):
    values = np.asarray(values, dtype=float)
    mask = np.isfinite(values)
    values = values[mask]
    if len(values) == 0:
        return np.nan
    angles = np.deg2rad(values)
    if weights is None:
        weights = np.ones(len(values), dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)[mask]
    sin_mean = np.sum(np.sin(angles) * weights)
    cos_mean = np.sum(np.cos(angles) * weights)
    return float((np.degrees(np.arctan2(sin_mean, cos_mean)) + 360) % 360)

def add_time_columns(df, time_col="time_utc", local_tz=LOCAL_TZ):
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col], utc=True)
    df["time_local"] = df[time_col].dt.tz_convert(local_tz)
    df["date_local"] = df["time_local"].dt.strftime("%Y-%m-%d")
    df["hour_local"] = df["time_local"].dt.hour
    df["year_local"] = df["time_local"].dt.year
    df["month_local"] = df["time_local"].dt.month
    df["day_local"] = df["time_local"].dt.day
    return df

def cyclic_time_features(times):
    t = pd.to_datetime(times)
    if isinstance(t, pd.Series):
        hour = t.dt.hour
        doy = t.dt.dayofyear
    else:
        hour = t.hour
        doy = t.dayofyear
    return pd.DataFrame({
        "hour_sin": np.sin(2 * np.pi * hour / 24),
        "hour_cos": np.cos(2 * np.pi * hour / 24),
        "doy_sin": np.sin(2 * np.pi * doy / 365.25),
        "doy_cos": np.cos(2 * np.pi * doy / 365.25),
    })

def detect_lat_lon_names(ds):
    lat_name = None
    lon_name = None
    for c in ["latitude", "lat"]:
        if c in ds.coords or c in ds.variables:
            lat_name = c
            break
    for c in ["longitude", "lon"]:
        if c in ds.coords or c in ds.variables:
            lon_name = c
            break
    if lat_name is None or lon_name is None:
        raise ValueError(f"Could not find lat/lon. Available: {list(ds.variables)}")
    return lat_name, lon_name

def convert_lon_360_to_180(ds, lon_name):
    ds = ds.assign_coords({lon_name: (((ds[lon_name] + 180) % 360) - 180)})
    return ds.sortby(lon_name)

def find_time_coord(ds):
    for c in ["valid_time", "time"]:
        if c in ds.coords:
            return c
    raise ValueError(f"No time coordinate found. Coords: {list(ds.coords)}")

def try_open_cfgrib(path, filter_by_keys=None):
    try:
        kwargs = {
            "engine": "cfgrib",
            "backend_kwargs": {"indexpath": "", "errors": "warn"}
        }
        if filter_by_keys is not None:
            kwargs["backend_kwargs"]["filter_by_keys"] = filter_by_keys
        return xr.open_dataset(path, **kwargs)
    except Exception:
        return None

def open_grib_candidate_sets(path):
    candidates = [
        {"typeOfLevel": "heightAboveGround", "level": 2},
        {"typeOfLevel": "heightAboveGround", "level": 10},
        {"typeOfLevel": "meanSea"},
        {"typeOfLevel": "surface"},
        {"typeOfLevel": "atmosphereSingleLayer"},
        {"typeOfLevel": "isobaricInhPa"},
    ]
    opened, seen = [], set()
    for filt in candidates:
        ds = try_open_cfgrib(path, filt)
        if ds is not None:
            sig = (tuple(sorted(ds.data_vars)), tuple(sorted(ds.coords)))
            if sig not in seen:
                seen.add(sig)
                opened.append((filt, ds))
    return opened

def select_point_dataarray(da, lat=TARGET_POINT_LAT, lon=TARGET_POINT_LON):
    ds_tmp = da.to_dataset(name="tmp")
    lat_name, lon_name = detect_lat_lon_names(ds_tmp)
    if lon_name in ds_tmp.coords:
        lon_vals = np.atleast_1d(ds_tmp[lon_name].values)
        if lon_vals.size > 0 and float(np.nanmax(lon_vals)) > 180:
            ds_tmp = convert_lon_360_to_180(ds_tmp, lon_name)
    return ds_tmp["tmp"].sel({lat_name: lat, lon_name: lon}, method="nearest")

def extract_point_dataset(ds, prefix="site"):
    out = {}
    for var in ["u10", "v10", "t2m", "d2m", "mslp", "blh", "tcc"]:
        if var in ds.data_vars:
            out[f"{prefix}_{var}"] = select_point_dataarray(ds[var])
    return xr.Dataset(out)

def direction_in_sector(direction_deg, sector_min, sector_max):
    direction_deg = np.asarray(direction_deg, dtype=float) % 360
    if sector_min <= sector_max:
        return (direction_deg >= sector_min) & (direction_deg <= sector_max)
    return (direction_deg >= sector_min) | (direction_deg <= sector_max)

## 1. Hämta ERA5 och bygg historik för punktplatsen

In [5]:
# Samma huvudstruktur som originalet, men nu för en punktfokuserad analoghistorik.
def download_era5_year(year):
    target_file = os.path.join(ERA5_DIR, f"era5_point_{year}_{ERA5_DATA_VERSION}.grib")
    if os.path.exists(target_file) and os.path.getsize(target_file) > 0:
        print("Exists:", target_file)
        return target_file

    c = cdsapi.Client()
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "10m_u_component_of_wind",
                "10m_v_component_of_wind",
                "2m_temperature",
                "2m_dewpoint_temperature",
                "mean_sea_level_pressure",
                "boundary_layer_height",
                "total_cloud_cover",
            ],
            "year": year,
            "month": TARGET_MONTH,
            "day": TARGET_DAYS,
            "time": [f"{h:02d}:00" for h in range(24)],
            "data_format": "grib",
            "download_format": "unarchived",
            "area": DOWNLOAD_AREA,
        },
        target_file,
    )
    print("Downloaded:", target_file)
    return target_file

era5_files = [download_era5_year(y) for y in TRAIN_YEARS]
print("ERA5 files:", len(era5_files))


Exists: /content/era5_point_analog\era5_point_2010_v1.grib
ERA5 files: 1


In [6]:
def open_era5(path):
    opened = open_grib_candidate_sets(path)
    if not opened:
        raise ValueError(f"Could not open ERA5 file: {path}")
    ds = xr.merge([d for _, d in opened], compat="override")
    rename_map = {}
    if "msl" in ds.data_vars:
        rename_map["msl"] = "mslp"
    return ds.rename(rename_map)

def build_era5_dataframe(era5_paths):
    all_frames = []
    for path in era5_paths:
        print("Reading ERA5:", path)
        ds = open_era5(path)
        point_ds = extract_point_dataset(ds, prefix="site")
        time_name = find_time_coord(point_ds)
        df = point_ds.to_dataframe().reset_index().rename(columns={time_name: "time_utc"})
        df = add_time_columns(df, "time_utc")
        all_frames.append(df)
    out = pd.concat(all_frames, ignore_index=True)
    out = out.sort_values("time_utc").drop_duplicates(subset=["time_utc"]).reset_index(drop=True)
    return out

era5_df = build_era5_dataframe(era5_files)
print("ERA5 rows:", len(era5_df))
display(era5_df.head())

Reading ERA5: /content/era5_point_analog\era5_point_2010_v1.grib
ERA5 rows: 3672


,time,number,step,surface,latitude,longitude,time_utc,site_u10,site_v10,site_t2m,site_d2m,site_mslp,site_blh,site_tcc,time_local,date_local,hour_local,year_local,month_local,day_local
0,2010-05-01 00:00:00,0,0 days,0.0,33.75,-118.25,2010-05-01 00:00:00+00:00,8.322372,0.394226,291.764893,277.616211,101059.3125,747.058105,0.0,2010-04-30 17:00:00-07:00,2010-04-30,17,2010,4,30
1,2010-05-01 01:00:00,0,0 days,0.0,33.75,-118.25,2010-05-01 01:00:00+00:00,8.256699,-0.065109,291.291748,277.521973,101021.0000,652.250488,0.0,2010-04-30 18:00:00-07:00,2010-04-30,18,2010,4,30
2,2010-05-01 02:00:00,0,0 days,0.0,33.75,-118.25,2010-05-01 02:00:00+00:00,7.293564,-0.264282,290.592529,278.772217,101018.4375,482.275879,0.0,2010-04-30 19:00:00-07:00,2010-04-30,19,2010,4,30
3,2010-05-01 03:00:00,0,0 days,0.0,33.75,-118.25,2010-05-01 03:00:00+00:00,6.045929,-1.108215,289.253967,280.213257,101033.5000,321.194763,0.0,2010-04-30 20:00:00-07:00,2010-04-30,20,2010,4,30
4,2010-05-01 04:00:00,0,0 days,0.0,33.75,-118.25,2010-05-01 04:00:00+00:00,4.664856,-1.461426,288.677124,281.634399,101043.7500,218.190323,0.0,2010-04-30 21:00:00-07:00,2010-04-30,21,2010,4,30


## 2. Hämta senaste tillgängliga GFS för samma punkt

In [7]:
import os
import requests
import pandas as pd

def download_rda_gfs_file(
    yyyymmdd,
    cycle="00",
    forecast_hour="000",
    out_dir=GFS_DIR,
    timeout=180,
    force=False,
):
    os.makedirs(out_dir, exist_ok=True)

    year = yyyymmdd[:4]
    file_name = f"gfs.0p25.{yyyymmdd}{cycle}.f{forecast_hour}.grib2"
    url = f"https://thredds.rda.ucar.edu/thredds/fileServer/files/g/d084001/{year}/{yyyymmdd}/{file_name}"
    out_path = os.path.join(out_dir, file_name)

    if (not force) and os.path.exists(out_path) and os.path.getsize(out_path) > 1024:
        try:
            with open(out_path, "rb") as f:
                start = f.read(20)
            if b"GRIB" in start:
                print("Reusing cached:", out_path)
                return out_path
        except Exception:
            pass

    print("Downloading:", url)
    r = requests.get(url, stream=True, timeout=timeout)
    if r.status_code != 200:
        raise RuntimeError(f"RDA download failed for {file_name} with status {r.status_code}")

    with open(out_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    with open(out_path, "rb") as f:
        start = f.read(20)

    if b"GRIB" not in start:
        if os.path.exists(out_path):
            os.remove(out_path)
        raise RuntimeError(f"Nedladdad fil verkar inte vara GRIB2: {file_name}")

    print(f"Saved: {out_path}")
    return out_path


def download_rda_gfs_date_range(
    start_date,
    end_date,
    cycle="00",
    forecast_hours=None,
    force=False,
):
    if forecast_hours is None:
        forecast_hours = ["000"]

    dates = pd.date_range(start=pd.to_datetime(start_date), end=pd.to_datetime(end_date), freq="D")
    paths = []

    for dt in dates:
        yyyymmdd = dt.strftime("%Y%m%d")
        print(f"\n=== Downloading date {yyyymmdd} cycle {cycle}z ===")
        for fh in forecast_hours:
            try:
                path = download_rda_gfs_file(
                    yyyymmdd=yyyymmdd,
                    cycle=cycle,
                    forecast_hour=fh,
                    out_dir=GFS_DIR,
                    force=FORCE_REDOWNLOAD_GFS,
                )
                paths.append(path)
            except Exception as e:
                print(f"Skipped {yyyymmdd} f{fh}: {e}")

    return paths


gfs_files = download_rda_gfs_date_range(
    start_date=RDA_GFS_START_DATE,
    end_date=RDA_GFS_END_DATE,
    cycle=RDA_GFS_CYCLE,
    forecast_hours=GFS_FORECAST_HOURS,
    force=FORCE_REDOWNLOAD_GFS,
)

print("Antal GFS-filer:", len(gfs_files))
print(gfs_files[:10])




=== Downloading date 20250601 cycle 00z ===
Reusing cached: /content/gfs_point_analog\gfs.0p25.2025060100.f015.grib2
Reusing cached: /content/gfs_point_analog\gfs.0p25.2025060100.f018.grib2

=== Downloading date 20250602 cycle 00z ===
Reusing cached: /content/gfs_point_analog\gfs.0p25.2025060200.f015.grib2
Downloading: https://thredds.rda.ucar.edu/thredds/fileServer/files/g/d084001/2025/20250602/gfs.0p25.2025060200.f018.grib2
Saved: /content/gfs_point_analog\gfs.0p25.2025060200.f018.grib2

=== Downloading date 20250603 cycle 00z ===
Reusing cached: /content/gfs_point_analog\gfs.0p25.2025060300.f015.grib2
Reusing cached: /content/gfs_point_analog\gfs.0p25.2025060300.f018.grib2

=== Downloading date 20250604 cycle 00z ===
Reusing cached: /content/gfs_point_analog\gfs.0p25.2025060400.f015.grib2
Reusing cached: /content/gfs_point_analog\gfs.0p25.2025060400.f018.grib2

=== Downloading date 20250605 cycle 00z ===
Reusing cached: /content/gfs_point_analog\gfs.0p25.2025060500.f015.grib2
Downl

In [8]:
# Robust GFS-läsning med samma grundidé som i originalet, men här extraheras en punktserie.
import re

def remove_old_idx(grib_path):
    for f in glob.glob(grib_path + "*.idx"):
        try:
            os.remove(f)
        except Exception:
            pass

TARGET_SPECS = {
    "u10": {
        "filters": [{"typeOfLevel": "heightAboveGround", "level": 10}],
        "exact_names": {"u10", "10u", "ugrd"},
        "must_have": ["10", "wind"],
        "forbid": ["dew", "dpt", "temp", "tmp", "kelvin", "msl", "pressure"],
        "expected_units": ["m s**-1", "m/s", "m s-1"],
    },
    "v10": {
        "filters": [{"typeOfLevel": "heightAboveGround", "level": 10}],
        "exact_names": {"v10", "10v", "vgrd"},
        "must_have": ["10", "wind"],
        "forbid": ["dew", "dpt", "temp", "tmp", "kelvin", "msl", "pressure"],
        "expected_units": ["m s**-1", "m/s", "m s-1"],
    },
    "t2m": {
        "filters": [{"typeOfLevel": "heightAboveGround", "level": 2}],
        "exact_names": {"t2m", "2t", "tmp"},
        "must_have": ["2"],
        "forbid": ["wind", "ugrd", "vgrd", "dew", "dpt"],
        "expected_units": ["k"],
    },
    "d2m": {
        "filters": [{"typeOfLevel": "heightAboveGround", "level": 2}],
        "exact_names": {"d2m", "2d", "dpt"},
        "must_have": ["2"],
        "forbid": ["wind", "ugrd", "vgrd", "temp", "tmp"],
        "expected_units": ["k"],
    },
    "mslp": {
        "filters": [{"typeOfLevel": "meanSea"}],
        "exact_names": {"msl", "mslp", "prmsl"},
        "must_have": ["sea"],
        "forbid": ["wind", "tmp", "dpt"],
        "expected_units": ["pa"],
    },
    "blh": {
        "filters": [{"typeOfLevel": "planetaryBoundaryLayer"}, {"typeOfLevel": "surface"}, {"typeOfLevel": "atmosphereSingleLayer"}],
        "exact_names": {"blh", "hpbl", "pblh"},
        "must_have": [],
        "forbid": ["wind", "tmp", "dpt", "pressure"],
        "expected_units": ["m"],
    },
    "tcc": {
        "filters": [{"typeOfLevel": "atmosphere"}, {"typeOfLevel": "surface"}, {"typeOfLevel": "atmosphereSingleLayer"}],
        "exact_names": {"tcc", "tcdc"},
        "must_have": ["cloud"],
        "forbid": ["wind", "tmp", "dpt", "pressure"],
        "expected_units": ["%", "1"],
    },
}

def _metadata_text_for_da(da):
    pieces = [str(da.name)]
    for k, v in da.attrs.items():
        pieces.append(f"{k}={v}")
    return " ".join(pieces).lower()

def _candidate_names_for_da(da):
    names = set()
    for key in ["cfVarName", "GRIB_cfVarName", "GRIB_shortName", "shortName", "standard_name", "long_name"]:
        val = da.attrs.get(key)
        if val is not None:
            names.add(str(val).lower())
    names.add(str(da.name).lower())
    return names

def _units_text(da):
    for key in ["units", "GRIB_units"]:
        val = da.attrs.get(key)
        if val is not None:
            return str(val).lower()
    return ""

def _physically_plausible_for_target(da, target):
    try:
        vals = np.asarray(da.values, dtype=float)
        finite = vals[np.isfinite(vals)]
        if finite.size == 0:
            return False
        p01 = float(np.nanpercentile(finite, 1))
        p99 = float(np.nanpercentile(finite, 99))
    except Exception:
        return True

    if target in ["u10", "v10"]:
        return (-120 <= p01 <= 120) and (-120 <= p99 <= 120)
    if target in ["t2m", "d2m"]:
        return 180 <= p01 <= 340 and 180 <= p99 <= 340
    if target == "mslp":
        return 70000 <= p01 <= 110000 and 70000 <= p99 <= 110000
    if target == "blh":
        return -10 <= p01 <= 10000 and -10 <= p99 <= 10000
    if target == "tcc":
        return (-0.1 <= p01 <= 110.0) and (-0.1 <= p99 <= 110.0)
    return True

def _score_da_for_target(da, target, filt):
    spec = TARGET_SPECS[target]
    names = _candidate_names_for_da(da)
    meta = _metadata_text_for_da(da)
    units = _units_text(da)
    score = 0
    if names & spec["exact_names"]:
        score += 300
    for token in spec["must_have"]:
        if token in meta:
            score += 40
    for token in spec["forbid"]:
        if token in meta:
            score -= 200
    if any(u in units for u in spec["expected_units"]):
        score += 80
    if not _physically_plausible_for_target(da, target):
        score -= 1000
    return score

def _pick_best_da_from_dataset(ds, target, filt):
    best = None
    best_score = -10**9
    for var in ds.data_vars:
        da = ds[var]
        score = _score_da_for_target(da, target, filt)
        if score > best_score:
            best_score = score
            best = da
    if best is None or best_score < 220:
        return None, best_score
    return best, best_score

def _open_target_da(path, target, debug=False):
    spec = TARGET_SPECS[target]
    best_overall = None
    best_score = -10**9
    for filt in spec["filters"]:
        ds = try_open_cfgrib(path, filt)
        if ds is None:
            continue
        da, score = _pick_best_da_from_dataset(ds, target, filt)
        if da is not None and score > best_score:
            best_overall = da
            best_score = score
    return best_overall

def normalize_da(da, desired_name):
    squeeze_dims = []
    for d in da.dims:
        if d not in ["time", "valid_time", "latitude", "longitude", "lat", "lon"]:
            if da.sizes.get(d, 1) == 1:
                squeeze_dims.append(d)
    if squeeze_dims:
        da = da.squeeze(squeeze_dims, drop=True)
    return da.rename(desired_name)

def build_gfs_dataframe_robust(gfs_paths, debug=False):
    rows = []
    for gfs_path in gfs_paths:
        print("\nProcessing GFS:", gfs_path)
        if not os.path.exists(gfs_path):
            continue
        remove_old_idx(gfs_path)
        found = {}
        for target in ["u10", "v10", "t2m", "d2m", "mslp", "blh", "tcc"]:
            da = _open_target_da(gfs_path, target, debug=debug)
            if da is not None:
                found[target] = normalize_da(da, target)

        if not {"u10", "v10", "t2m"}.issubset(found.keys()):
            print("Missing key variables in:", gfs_path)
            continue

        row = {"gfs_source_vars_found": ",".join(sorted(found.keys()))}
        time_source = found["u10"]
        time_name = "valid_time" if "valid_time" in time_source.coords else "time"
        row["time_utc"] = pd.to_datetime(np.ravel(time_source[time_name].values)[0], utc=True)

        for key, da in found.items():
            point_val = select_point_dataarray(da)
            flat = np.ravel(point_val.values)
            row[f"site_{key}"] = float(flat[0]) if flat.size else np.nan

        rows.append(row)

    out = pd.DataFrame(rows)
    out = add_time_columns(out, "time_utc")
    out = out.sort_values("time_utc").drop_duplicates(subset=["time_utc"]).reset_index(drop=True)
    return out

gfs_df = build_gfs_dataframe_robust(gfs_files, debug=False)
print("GFS rows:", len(gfs_df))
print("GFS columns:", list(gfs_df.columns))
if gfs_df.empty:
    raise ValueError("GFS kunde inte läsas in korrekt. Kontrollera att u10/v10/t2m verkligen hittas.")
display(gfs_df.head())


Processing GFS: /content/gfs_point_analog\gfs.0p25.2025060100.f015.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025060100.f018.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025060200.f015.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025060200.f018.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025060300.f015.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025060300.f018.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025060400.f015.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025060400.f018.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025060500.f015.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025060500.f018.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025060600.f015.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025060600.f018.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025060700.f015.grib2

Processing GFS: /content/gfs_point_an

skipping corrupted Message
Traceback (most recent call last):
  File "C:\Users\ollea\anaconda3\Lib\site-packages\cfgrib\messages.py", line 274, in itervalues
    yield self.filestream.message_from_file(file, errors=errors)
          ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ollea\anaconda3\Lib\site-packages\cfgrib\messages.py", line 341, in message_from_file
    return Message.from_file(file, offset, **kwargs)
           ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ollea\anaconda3\Lib\site-packages\cfgrib\messages.py", line 97, in from_file
    codes_id = eccodes.codes_grib_new_from_file(file)
  File "C:\Users\ollea\anaconda3\Lib\site-packages\gribapi\gribapi.py", line 415, in grib_new_from_file
    GRIB_CHECK(err)
    ~~~~~~~~~~^^^^^
  File "C:\Users\ollea\anaconda3\Lib\site-packages\gribapi\gribapi.py", line 232, in GRIB_CHECK
    errors.raise_grib_error(errid)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^
  File "C:\Users\ollea\anaconda3\Lib\site-pac

Missing key variables in: /content/gfs_point_analog\gfs.0p25.2025060900.f015.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025060900.f018.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025061000.f015.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025061000.f018.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025061100.f015.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025061100.f018.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025061200.f015.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025061200.f018.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025061300.f015.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025061300.f018.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025061400.f015.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025061400.f018.grib2

Processing GFS: /content/gfs_point_analog\gfs.0p25.2025061500.f015.grib2

Processing GFS: /content/gfs

,gfs_source_vars_found,time_utc,site_u10,site_v10,site_t2m,site_d2m,site_mslp,time_local,date_local,hour_local,year_local,month_local,day_local
0,"d2m,mslp,t2m,u10,v10",2025-06-01 15:00:00+00:00,0.780513,-2.564595,293.602570,288.151276,101315.898438,2025-06-01 08:00:00-07:00,2025-06-01,8,2025,6,1
1,"d2m,mslp,t2m,u10,v10",2025-06-01 18:00:00+00:00,1.579128,-1.249473,297.804047,286.051270,101237.976562,2025-06-01 11:00:00-07:00,2025-06-01,11,2025,6,1
2,"d2m,mslp,t2m,u10,v10",2025-06-02 15:00:00+00:00,-1.164064,3.744915,290.200012,287.200012,101480.953125,2025-06-02 08:00:00-07:00,2025-06-02,8,2025,6,2
3,"d2m,mslp,t2m,u10,v10",2025-06-02 18:00:00+00:00,1.311389,3.484656,291.138611,287.056152,101556.640625,2025-06-02 11:00:00-07:00,2025-06-02,11,2025,6,2
4,"d2m,mslp,t2m,u10,v10",2025-06-03 15:00:00+00:00,1.715100,1.018745,290.032288,285.799988,101448.500000,2025-06-03 08:00:00-07:00,2025-06-03,8,2025,6,3


## 3. Punktbaserade features och sjöbrisfilter från ERA5

In [9]:
def add_physical_features(df):
    df = df.copy()
    if "site_u10" in df.columns and "site_v10" in df.columns:
        df["site_wind_speed"] = wind_speed(df["site_u10"], df["site_v10"])
        df["site_wind_dir"] = wind_dir_deg(df["site_u10"], df["site_v10"])
        radians = np.deg2rad(df["site_wind_dir"])
        df["site_wind_dir_sin"] = np.sin(radians)
        df["site_wind_dir_cos"] = np.cos(radians)
    if "site_t2m" in df.columns:
        df["site_t2m_C"] = df["site_t2m"] - 273.15
    if "site_d2m" in df.columns:
        df["site_d2m_C"] = df["site_d2m"] - 273.15

    if not {"hour_sin", "hour_cos", "doy_sin", "doy_cos"}.issubset(df.columns):
        tf = cyclic_time_features(df["time_local"])
        df = pd.concat([df.reset_index(drop=True), tf.reset_index(drop=True)], axis=1)

    for col in ["site_wind_speed", "site_t2m_C", "site_mslp", "site_blh", "site_tcc"]:
        if col in df.columns:
            df[f"{col}_diff1"] = df[col].diff()
            df[f"{col}_diff3"] = df[col].diff(3)

    if "site_wind_dir" in df.columns:
        df["site_wind_dir_diff1"] = circular_diff_deg(df["site_wind_dir"], df["site_wind_dir"].shift(1))
        df["site_wind_dir_diff3"] = circular_diff_deg(df["site_wind_dir"], df["site_wind_dir"].shift(3))

    return df

def create_daily_sea_breeze_flag(day_df):
    day_df = day_df.sort_values("time_local").copy()
    if ANALOG_MATCH_LOCAL_HOUR not in day_df["hour_local"].values:
        return {
            "sea_breeze_day": 0,
            "match_hour_present": 0,
        }

    row_10 = day_df[day_df["hour_local"] == ANALOG_MATCH_LOCAL_HOUR].iloc[0]
    aft = day_df[day_df["hour_local"].isin(FORECAST_LOCAL_HOURS)].copy()
    if aft.empty:
        return {
            "sea_breeze_day": 0,
            "match_hour_present": 0,
        }

    max_speed_increase = float(aft["site_wind_speed"].max() - row_10["site_wind_speed"]) if "site_wind_speed" in aft.columns else np.nan
    max_temp_rise = float(aft["site_t2m_C"].max() - row_10["site_t2m_C"]) if "site_t2m_C" in aft.columns else np.nan
    max_dir_shift = float(np.nanmax(np.abs(circular_diff_deg(aft["site_wind_dir"], row_10["site_wind_dir"])))) if "site_wind_dir" in aft.columns else np.nan
    onshore_fraction = float(direction_in_sector(aft["site_wind_dir"], SEA_BREEZE_ONSHORE_DIR_MIN, SEA_BREEZE_ONSHORE_DIR_MAX).mean()) if "site_wind_dir" in aft.columns else 0.0

    cond_speed = np.isfinite(max_speed_increase) and max_speed_increase >= SEA_BREEZE_MIN_SPEED_INCREASE
    cond_dir = np.isfinite(max_dir_shift) and max_dir_shift >= SEA_BREEZE_MIN_DIR_SHIFT_DEG
    cond_onshore = onshore_fraction >= 0.5
    cond_temp = np.isfinite(max_temp_rise) and max_temp_rise >= SEA_BREEZE_MIN_TEMP_RISE_C

    sea_breeze_day = int(cond_speed and cond_dir and cond_onshore and cond_temp)
    return {
        "sea_breeze_day": sea_breeze_day,
        "match_hour_present": 1,
        "max_speed_increase_ms": max_speed_increase,
        "max_temp_rise_C": max_temp_rise,
        "max_dir_shift_deg": max_dir_shift,
        "onshore_fraction_11_16": onshore_fraction,
    }

era5_df = add_physical_features(era5_df)
gfs_df = add_physical_features(gfs_df)

print("ERA5 columns:", len(era5_df.columns))
print("GFS columns:", len(gfs_df.columns))
display(era5_df.head())

ERA5 columns: 42
GFS columns: 31


,time,number,step,surface,latitude,longitude,time_utc,site_u10,site_v10,site_t2m,...,site_t2m_C_diff1,site_t2m_C_diff3,site_mslp_diff1,site_mslp_diff3,site_blh_diff1,site_blh_diff3,site_tcc_diff1,site_tcc_diff3,site_wind_dir_diff1,site_wind_dir_diff3
0,2010-05-01 00:00:00,0,0 days,0.0,33.75,-118.25,2010-05-01 00:00:00+00:00,8.322372,0.394226,291.764893,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-05-01 01:00:00,0,0 days,0.0,33.75,-118.25,2010-05-01 01:00:00+00:00,8.256699,-0.065109,291.291748,...,-0.473145,NaN,-38.3125,NaN,-94.807617,NaN,0.0,NaN,3.163849,NaN
2,2010-05-01 02:00:00,0,0 days,0.0,33.75,-118.25,2010-05-01 02:00:00+00:00,7.293564,-0.264282,290.592529,...,-0.699219,NaN,-2.5625,NaN,-169.974609,NaN,0.0,NaN,1.623383,NaN
3,2010-05-01 03:00:00,0,0 days,0.0,33.75,-118.25,2010-05-01 03:00:00+00:00,6.045929,-1.108215,289.253967,...,-1.338562,-2.510925,15.0625,-25.8125,-161.081116,-425.863342,0.0,0.0,8.311768,13.098999
4,2010-05-01 04:00:00,0,0 days,0.0,33.75,-118.25,2010-05-01 04:00:00+00:00,4.664856,-1.461426,288.677124,...,-0.576843,-2.614624,10.2500,22.7500,-103.004440,-434.060181,0.0,0.0,7.007996,16.943146


## 4. Bygg historiskt bibliotek av endast ERA5-fall med sjöbris

In [10]:
def build_day_feature_row(day_df):
    day_df = day_df.sort_values("time_local").copy()
    row_10 = day_df[day_df["hour_local"] == ANALOG_MATCH_LOCAL_HOUR]
    if row_10.empty:
        return None
    row_10 = row_10.iloc[0]

    morning = day_df[day_df["hour_local"].between(8, ANALOG_MATCH_LOCAL_HOUR)].copy()
    aft = day_df[day_df["hour_local"].isin(FORECAST_LOCAL_HOURS)].copy()
    if aft.empty:
        return None

    out = {
        "date": str(day_df["date_local"].iloc[0]),
        "match_hour_local": ANALOG_MATCH_LOCAL_HOUR,
        "site_u10_10": float(row_10["site_u10"]),
        "site_v10_10": float(row_10["site_v10"]),
        "site_t2m_C_10": float(row_10["site_t2m_C"]) if "site_t2m_C" in row_10 else np.nan,
        "site_wind_speed_10": float(row_10["site_wind_speed"]) if "site_wind_speed" in row_10 else np.nan,
        "site_wind_dir_10": float(row_10["site_wind_dir"]) if "site_wind_dir" in row_10 else np.nan,
        "site_wind_dir_sin_10": float(row_10["site_wind_dir_sin"]) if "site_wind_dir_sin" in row_10 else np.nan,
        "site_wind_dir_cos_10": float(row_10["site_wind_dir_cos"]) if "site_wind_dir_cos" in row_10 else np.nan,
        "site_wind_speed_diff1_10": float(row_10["site_wind_speed_diff1"]) if "site_wind_speed_diff1" in row_10 else np.nan,
        "site_t2m_C_diff1_10": float(row_10["site_t2m_C_diff1"]) if "site_t2m_C_diff1" in row_10 else np.nan,
        "site_wind_dir_diff1_10": float(row_10["site_wind_dir_diff1"]) if "site_wind_dir_diff1" in row_10 else np.nan,
        "morning_temp_mean": float(morning["site_t2m_C"].mean()) if "site_t2m_C" in morning.columns else np.nan,
        "morning_wind_speed_mean": float(morning["site_wind_speed"].mean()) if "site_wind_speed" in morning.columns else np.nan,
        "morning_wind_dir_mean": float(circular_mean_deg(morning["site_wind_dir"])) if "site_wind_dir" in morning.columns else np.nan,
        "forecast_speed_mean_11_16": float(aft["site_wind_speed"].mean()) if "site_wind_speed" in aft.columns else np.nan,
        "forecast_speed_max_11_16": float(aft["site_wind_speed"].max()) if "site_wind_speed" in aft.columns else np.nan,
        "forecast_dir_mean_11_16": float(circular_mean_deg(aft["site_wind_dir"])) if "site_wind_dir" in aft.columns else np.nan,
    }

    hourly_records = []
    for _, hr in aft.iterrows():
        hourly_records.append({
            "hour_local": int(hr["hour_local"]),
            "wind_speed": float(hr["site_wind_speed"]) if "site_wind_speed" in hr else np.nan,
            "wind_dir": float(hr["site_wind_dir"]) if "site_wind_dir" in hr else np.nan,
            "temp_C": float(hr["site_t2m_C"]) if "site_t2m_C" in hr else np.nan,
        })
    out["hourly_outcomes"] = hourly_records

    out.update(create_daily_sea_breeze_flag(day_df))
    return out

def build_era5_analog_library(era5_df):
    rows = []
    for date_local, day_df in era5_df.groupby("date_local"):
        payload = build_day_feature_row(day_df)
        if payload is not None:
            rows.append(payload)
    library = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    sea_breeze_only = library[library["sea_breeze_day"] == 1].copy().reset_index(drop=True)
    return library, sea_breeze_only

era5_daily_library_all, era5_daily_library_sb = build_era5_analog_library(era5_df)

print("Alla ERA5-dagar i biblioteket:", len(era5_daily_library_all))
print("ERA5-dagar med sjöbris:", len(era5_daily_library_sb))
display(era5_daily_library_sb.head())

Alla ERA5-dagar i biblioteket: 153
ERA5-dagar med sjöbris: 78


,date,match_hour_local,site_u10_10,site_v10_10,site_t2m_C_10,site_wind_speed_10,site_wind_dir_10,site_wind_dir_sin_10,site_wind_dir_cos_10,site_wind_speed_diff1_10,...,forecast_speed_mean_11_16,forecast_speed_max_11_16,forecast_dir_mean_11_16,hourly_outcomes,sea_breeze_day,match_hour_present,max_speed_increase_ms,max_temp_rise_C,max_dir_shift_deg,onshore_fraction_11_16
0,2010-05-01,9,-0.160385,1.636566,15.733063,1.644406,174.402832,0.097534,-0.995232,0.038096,...,5.642803,6.892998,229.090313,"[{'hour_local': 11, 'wind_speed': 3.4465856552...",1,1,5.248591,3.853516,81.643768,1.0
1,2010-05-03,9,-0.388550,0.321976,16.938385,0.504618,129.647186,0.769988,-0.638058,0.028974,...,3.291178,4.774422,235.995665,"[{'hour_local': 11, 'wind_speed': 1.0796732902...",1,1,4.269804,5.311279,125.555664,1.0
2,2010-05-04,9,0.276627,1.474976,16.461945,1.500692,190.622238,-0.184333,-0.982864,0.483941,...,4.127311,4.891892,222.536604,"[{'hour_local': 11, 'wind_speed': 3.2854781150...",1,1,3.391201,3.885132,42.323578,1.0
3,2010-05-05,9,0.035858,1.651215,16.625275,1.651604,181.244049,-0.021711,-0.999764,-0.001171,...,4.218533,5.178677,213.438957,"[{'hour_local': 11, 'wind_speed': 2.9074161052...",1,1,3.527073,2.882202,43.556427,1.0
4,2010-05-06,9,-0.462189,2.338608,16.610260,2.383842,168.820465,0.193884,-0.981024,-0.132142,...,4.047884,4.939064,220.226141,"[{'hour_local': 11, 'wind_speed': 3.0626330375...",1,1,2.555221,2.200684,81.349091,1.0


## 5. Ta GFS-indatan kl 10 lokal tid och matcha mot historiska sjöbrisfall

In [11]:
ANALOG_FEATURE_COLUMNS = [
    "site_u10_10",
    "site_v10_10",
    "site_t2m_C_10",
    "site_wind_speed_10",
    "site_wind_dir_sin_10",
    "site_wind_dir_cos_10",
    "site_wind_speed_diff1_10",
    "site_t2m_C_diff1_10",
    "site_wind_dir_diff1_10",
    "morning_temp_mean",
    "morning_wind_speed_mean",
]

def build_day_feature_row(day_df):
    day_df = day_df.sort_values("time_local").copy()

    row_match = day_df[day_df["hour_local"] == ANALOG_MATCH_LOCAL_HOUR]
    if row_match.empty:
        return None
    row_match = row_match.iloc[0]

    morning = day_df[day_df["hour_local"].between(8, ANALOG_MATCH_LOCAL_HOUR)].copy()
    aft = day_df[day_df["hour_local"].isin(FORECAST_LOCAL_HOURS)].copy()
    if aft.empty:
        return None

    out = {
        "date": str(day_df["date_local"].iloc[0]),
        "match_hour_local": int(row_match["hour_local"]),
        "site_u10_10": float(row_match["site_u10"]),
        "site_v10_10": float(row_match["site_v10"]),
        "site_t2m_C_10": float(row_match["site_t2m_C"]) if "site_t2m_C" in row_match else np.nan,
        "site_wind_speed_10": float(row_match["site_wind_speed"]) if "site_wind_speed" in row_match else np.nan,
        "site_wind_dir_10": float(row_match["site_wind_dir"]) if "site_wind_dir" in row_match else np.nan,
        "site_wind_dir_sin_10": float(row_match["site_wind_dir_sin"]) if "site_wind_dir_sin" in row_match else np.nan,
        "site_wind_dir_cos_10": float(row_match["site_wind_dir_cos"]) if "site_wind_dir_cos" in row_match else np.nan,
        "site_wind_speed_diff1_10": float(row_match["site_wind_speed_diff1"]) if "site_wind_speed_diff1" in row_match else np.nan,
        "site_t2m_C_diff1_10": float(row_match["site_t2m_C_diff1"]) if "site_t2m_C_diff1" in row_match else np.nan,
        "site_wind_dir_diff1_10": float(row_match["site_wind_dir_diff1"]) if "site_wind_dir_diff1" in row_match else np.nan,
        "morning_temp_mean": float(morning["site_t2m_C"].mean()) if "site_t2m_C" in morning.columns else np.nan,
        "morning_wind_speed_mean": float(morning["site_wind_speed"].mean()) if "site_wind_speed" in morning.columns else np.nan,
        "forecast_speed_mean_11_16": float(aft["site_wind_speed"].mean()) if "site_wind_speed" in aft.columns else np.nan,
        "forecast_speed_max_11_16": float(aft["site_wind_speed"].max()) if "site_wind_speed" in aft.columns else np.nan,
        "forecast_dir_mean_11_16": float(circular_mean_deg(aft["site_wind_dir"])) if "site_wind_dir" in aft.columns else np.nan,
    }

    hourly_records = []
    for _, hr in aft.iterrows():
        hourly_records.append({
            "hour_local": int(hr["hour_local"]),
            "wind_speed": float(hr["site_wind_speed"]) if "site_wind_speed" in hr else np.nan,
            "wind_dir": float(hr["site_wind_dir"]) if "site_wind_dir" in hr else np.nan,
            "temp_C": float(hr["site_t2m_C"]) if "site_t2m_C" in hr else np.nan,
        })
    out["hourly_outcomes"] = hourly_records

    return out

def select_gfs_match_day(gfs_df, selected_date=SELECTED_GFS_DATE):
    candidates = []
    for date_local, day_df in gfs_df.groupby("date_local"):
        if ANALOG_MATCH_LOCAL_HOUR in day_df["hour_local"].values:
            candidates.append(str(date_local))

    if not candidates:
        raise ValueError(
            f"Ingen GFS-dag innehåller kl {ANALOG_MATCH_LOCAL_HOUR} lokal tid. "
            f"Tillgängliga lokala timmar i GFS: {sorted(gfs_df['hour_local'].dropna().unique().tolist())}"
        )

    chosen = candidates[0] if selected_date is None else selected_date
    if chosen not in candidates:
        raise ValueError(f"Vald GFS-dag {chosen} saknar kl {ANALOG_MATCH_LOCAL_HOUR}. Tillgängliga: {candidates}")

    out = gfs_df[gfs_df["date_local"] == chosen].copy().sort_values("time_local").reset_index(drop=True)
    return chosen, out

selected_gfs_date, gfs_day_df = select_gfs_match_day(gfs_df, selected_date=SELECTED_GFS_DATE)
gfs_day_features = pd.DataFrame([build_day_feature_row(gfs_day_df)])
print("Selected GFS date:", selected_gfs_date)
display(gfs_day_features.T)

def compute_distances(hist_df, target_df, feature_cols, method="euclidean"):
    X_hist = hist_df[feature_cols].copy()
    X_target = target_df[feature_cols].copy()

    X_hist = X_hist.fillna(X_hist.median(numeric_only=True))
    X_target = X_target.fillna(X_hist.median(numeric_only=True))

    scaler = StandardScaler()
    X_hist_scaled = scaler.fit_transform(X_hist)
    X_target_scaled = scaler.transform(X_target)[0]

    if method == "cosine":
        hist_norm = np.linalg.norm(X_hist_scaled, axis=1, keepdims=True)
        target_norm = np.linalg.norm(X_target_scaled.reshape(1, -1), axis=1, keepdims=True)
        hist_norm[hist_norm == 0] = 1.0
        target_norm[target_norm == 0] = 1.0
        sims = (X_hist_scaled / hist_norm) @ (X_target_scaled.reshape(-1, 1) / target_norm[0, 0])
        return 1.0 - sims[:, 0]

    return np.linalg.norm(X_hist_scaled - X_target_scaled, axis=1)

analog_candidates = era5_daily_library_sb.copy()
analog_candidates["distance"] = compute_distances(
    analog_candidates,
    gfs_day_features,
    ANALOG_FEATURE_COLUMNS,
    method=ANALOG_DISTANCE_METHOD,
)

analog_candidates = analog_candidates.sort_values("distance").reset_index(drop=True)
top_analogs = analog_candidates.head(ANALOG_TOP_K).copy()
top_analogs["rank"] = np.arange(1, len(top_analogs) + 1)
top_analogs["weight"] = 1.0 / np.maximum(top_analogs["distance"], 1e-6)
top_analogs["weight"] = top_analogs["weight"] / top_analogs["weight"].sum()

display(top_analogs[[
    "rank", "date", "distance", "weight",
    "match_hour_local",
    "max_speed_increase_ms", "max_temp_rise_C",
    "max_dir_shift_deg", "onshore_fraction_11_16"
]].round(3))


ValueError: Ingen GFS-dag innehåller kl 9 lokal tid. Tillgängliga lokala timmar i GFS: [8, 11]

## 6. Skapa prognos från vanligaste historiska utfall 11-16 lokal tid

In [ ]:
def direction_to_sector(direction_deg):
    if not np.isfinite(direction_deg):
        return "okänd"
    sectors = ["N", "NO", "O", "SO", "S", "SV", "V", "NV"]
    idx = int(((direction_deg % 360) + 22.5) // 45) % 8
    return sectors[idx]

def weighted_mode(values, weights):
    table = pd.DataFrame({"value": values, "weight": weights})
    table = table.dropna()
    if table.empty:
        return np.nan
    agg = table.groupby("value", as_index=False)["weight"].sum().sort_values("weight", ascending=False)
    return agg.iloc[0]["value"]

def build_forecast_from_analogs(top_analogs, forecast_hours=FORECAST_LOCAL_HOURS):
    rows = []
    for hour in forecast_hours:
        speed_vals = []
        dir_vals = []
        sector_vals = []
        weights = []

        for _, analog in top_analogs.iterrows():
            records = analog["hourly_outcomes"]
            matched = [r for r in records if int(r["hour_local"]) == int(hour)]
            if not matched:
                continue
            rec = matched[0]
            speed_vals.append(rec["wind_speed"])
            dir_vals.append(rec["wind_dir"])
            sector_vals.append(direction_to_sector(rec["wind_dir"]))
            weights.append(float(analog["weight"]))

        speed_vals = np.asarray(speed_vals, dtype=float)
        dir_vals = np.asarray(dir_vals, dtype=float)
        weights = np.asarray(weights, dtype=float)

        if len(speed_vals) == 0:
            rows.append({
                "hour_local": hour,
                "analog_speed_median": np.nan,
                "analog_speed_p25": np.nan,
                "analog_speed_p75": np.nan,
                "analog_dir_mean_deg": np.nan,
                "analog_dir_mode_sector": np.nan,
                "n_cases": 0,
            })
            continue

        rows.append({
            "hour_local": hour,
            "analog_speed_median": float(np.nanmedian(speed_vals)),
            "analog_speed_p25": float(np.nanpercentile(speed_vals, 25)),
            "analog_speed_p75": float(np.nanpercentile(speed_vals, 75)),
            "analog_dir_mean_deg": float(circular_mean_deg(dir_vals, weights=weights)),
            "analog_dir_mode_sector": weighted_mode(sector_vals, weights),
            "n_cases": int(len(speed_vals)),
        })

    forecast_df = pd.DataFrame(rows)
    forecast_df["analog_dir_mean_sector"] = forecast_df["analog_dir_mean_deg"].apply(direction_to_sector)
    return forecast_df

analog_forecast_df = build_forecast_from_analogs(top_analogs)
print("Prognosen nedan bygger bara på historiska ERA5-dagar som klassats som sjöbrisdagar.")
display(analog_forecast_df.round(2))

## 7. Visualisering av punktprognosen

In [ ]:
def plot_point_analog_forecast(gfs_day_df, analog_forecast_df):
    gfs_eval = gfs_day_df[gfs_day_df["hour_local"].isin(FORECAST_LOCAL_HOURS)].copy().sort_values("hour_local")

    plt.figure(figsize=(10, 4))
    plt.plot(analog_forecast_df["hour_local"], analog_forecast_df["analog_speed_median"], marker="o", label="Analog medianvind")
    plt.fill_between(
        analog_forecast_df["hour_local"],
        analog_forecast_df["analog_speed_p25"],
        analog_forecast_df["analog_speed_p75"],
        alpha=0.25,
        label="Historisk spridning P25-P75",
    )
    if "site_wind_speed" in gfs_eval.columns:
        plt.plot(gfs_eval["hour_local"], gfs_eval["site_wind_speed"], linestyle="--", label="Rå GFS vid punkten")
    plt.xlabel("Lokal timme")
    plt.ylabel("m/s")
    plt.title(f"Punktprognos för {selected_gfs_date} vid {TARGET_POINT_LAT:.4f}, {TARGET_POINT_LON:.4f}")
    plt.grid(True)
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(analog_forecast_df["hour_local"], analog_forecast_df["analog_dir_mean_deg"], marker="o", label="Analog riktning")
    if "site_wind_dir" in gfs_eval.columns:
        plt.plot(gfs_eval["hour_local"], gfs_eval["site_wind_dir"], linestyle="--", label="Rå GFS-riktning")
    plt.xlabel("Lokal timme")
    plt.ylabel("grader")
    plt.title("Historiskt vanligaste riktning efter analogmatchning")
    plt.grid(True)
    plt.legend()
    plt.show()

plot_point_analog_forecast(gfs_day_df, analog_forecast_df)

## 8. Nästa steg: vindstationer och ML

In [ ]:
# Nästa steg är att koppla detta analogflöde till vindstationer:
# 1. Läs in observationer från station/boj för samma punkt eller närmaste representativa station.
# 2. Jämför rå GFS mot analogprognosen timme för timme.
# 3. Träna sedan en separat ML-modell ovanpå:
#    - GFS-features kl 10 lokal tid
#    - analogavstånd och top-k-analogernas spridning
#    - analogprognosens median/procentiler 11-16
# Målet är då att förbättra den analogbaserade prognosen, inte att ersätta den som huvudmetod.

print("Klart: första analogsteget är nu punktbaserat och filtrerar först fram endast ERA5-fall med sjöbris.")